# Exploring HybridTopologyProtocol settings


The `HybridTopologyProtocol` exposes a number of simulation settings that can effect the convergence properties, accuracy and precision of an alchemical transformation. In this notebook, we will explore changes to common settings, explaining when you might want to consider changing them, the expected effect of changing them, and how to change them in practice.

## Inspecting the default settings

The default settings for the `HybridTopologyProtocol` are designed to be a good starting point for most alchemical transformations. However, depending on the specific system and transformation being studied, you may want to adjust these settings to improve convergence or precision of the estimation.

The default settings can be inspected by:

In [24]:
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()

For a complete explanation of each setting and how to customize the object see [Choose and Configure a Protocol](https://docs.openfree.energy/en/latest/cookbook/choose_protocol.html).

The main parameters that determine the amount of sampling performed and the associated cost for the hybrid topology protocol are:

- **Lambda windows**: the number of discrete alchemical states simulated during the transformation.
- **Production length**: the length of the production simulation performed at each lambda window.
- **Equilibration length**: the length of the equilibration simulation performed at each lambda window.
- **Protocol repeats**: the number of independent executions of the protocol.

The appropriate changes to these settings are transformation dependent with no single setting guarantying sufficient convergence or accuracy in general. These settings are good starting point to investigate the effect of additional sampling on the transformation under study however, but will also increase the computational cost of the simulation. The following example retrieves them from the default settings:

In [32]:
n_repeats = settings.protocol_repeats
n_windows = settings.lambda_settings.lambda_windows
production_length = settings.simulation_settings.production_length
equilibration_length = settings.simulation_settings.equilibration_length

print(f"Lambda windows: {n_windows}")
print(f"Production length per state: {production_length.m} ns")
print(f"Equilibration length per state: {equilibration_length.m} ns")
print(f"Independent repeats: {n_repeats}")

Lambda windows: 11
Production length per state: 5.0 ns
Equilibration length per state: 1.0 ns
Independent repeats: 3


## Estimating the computational cost of a protocol

A useful first approximation of the computational cost of a protocol execution is the total simulation time in nanoseconds across all lambda windows and repeats. This can be calculated as:

$$T_{total} = N_{repeats} \times N_{\lambda windows} \times (T_{production} + T_{equilibration})$$

So for the default settings the total simulation time is:

In [29]:
total_sim_time = n_repeats * n_windows * (production_length + equilibration_length)
print(f"Total simulation time: {total_sim_time.m} ns")

Total simulation time: 198.0 ns


The HybridTopologyProtocol simulates the complex and solvent legs of a transformation independently, so if both use the same settings the total cost of the protocol is double the above estimate. It is useful to consider this estimate when deciding how to adjust the settings for a given transformation.

# Increasing simulation length

This section describes how to increase the amount of production simulation performed at each lambda window and estimate the increase in computational cost.

## When to increase simulation length

Consider increasing the simulation length **before** running the simulation when:

- The transformation involves large structural changes due to for example changes in formal charge, you may also need to consider increasing the number of lambda windows see the next section for more information.

Consider increasing the simulation length **after** running the simulation when:

- Free-energy estimates differ substantially between independent repeats of the protocol.
- Forward and reverse free-energy estimates disagree by more than 1kbT at 50% of the simulation length and their uncertainties do not overlap.
- A large number of samples are discarded during analysis due to the automated equilibration and decorrelation methods.
- Replica exchange or MBAR analysis indicates that alchemical overlap or replica mixing is poor between lambda windows, you may also need to consider increasing the number of lambda windows see the next section for more information.

This information is available through the automated analysis performed by the `HybridTopologyProtocol` see the list of available [analysis metrics](https://docs.openfree.energy/en/latest/guide/protocols/relativehybridtopology.html#analysis) for more information, it is recommended to review the analysis metrics before deciding to increase the simulation length.

## When not to increase simulation length

Increasing the simulation length is not always the best solution. For example poor overlap between neighboring lambda windows may require changes to the number of lambda windows or the spacing between them.

## Changing the production length

The following example shows how to change the production length for a `HybridTopologyProtocol` object. In this example, we will double the production length from `5 ns` to `10 ns` per lambda window. Note the use of units from the `openff.units` package to specify the value is in nanoseconds. We will then calculate the new total simulation time and compare it to the default settings.


In [31]:
from openff.units import unit

long_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the production length to 10 ns
long_settings.simulation_settings.production_length = 10 * unit.nanoseconds

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=long_settings)

print(f"Updated production length: {protocol.settings.simulation_settings.production_length.m} ns")

# Calculate the new total simulation time
new_total_sim_time = n_repeats * n_windows * (protocol.settings.simulation_settings.production_length + protocol.settings.simulation_settings.equilibration_length)
print(f"New total simulation time: {new_total_sim_time.m} ns")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time - total_sim_time) / total_sim_time) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated production length: 10 ns
New total simulation time: 363.0 ns
Percentage increase in total simulation time: 83.33%


# Increasing the number of lambda windows

Increasing the number of lambda windows introduces additional intermediate alchemical states along the transformation which can make the pathway easier to traverse and improve the overlap between neighboring states. This section describes how to increase the number of lambda windows and estimate the increase in computational cost.

## When to increase the number of lambda windows

Consider increasing the number of lambda windows **before** running the simulation when:

- The transformation involves a change in net formal charge or substantial redistribution of charge, you may also need to consider increasing the simulation length see the previous section for more information.

Consider increasing the number of lambda windows **after** running the simulation when:

- Neighboring lambda windows have poor phase-space overlap or replica mixing statistics.

## When not to increase the number of lambda windows

Increasing the number of lambda windows is not always the best solution, slow changes in ligand conformation or binding site rearrangements may require longer simulation lengths to achieve convergence. In these cases, increasing the number of lambda windows may not improve convergence and will increase the computational cost of the protocol.

## Changing the number of lambda windows

The following example shows how to change the number of lambda windows for a `HybridTopologyProtocol` object. In this example, we will double the number of lambda windows from `11` to `22`. We will then calculate the new total simulation time and compare it to the default settings. Note the `lambda_windows` setting only accepts integer values and generates a uniform spacing of lambda values between `0` and `1`.


In [28]:
more_window_settings = RelativeHybridTopologyProtocol.default_settings()

# Increase the number of lambda windows to 22
more_window_settings.lambda_settings.lambda_windows = 22

# Create a new protocol object with the updated settings
protocol = RelativeHybridTopologyProtocol(settings=more_window_settings)

print(f"Updated number of lambda windows: {protocol.settings.lambda_settings.lambda_windows}")

# Calculate the new total simulation time
new_total_sim_time = n_repeats * protocol.settings.lambda_settings.lambda_windows * (protocol.settings.simulation_settings.production_length + protocol.settings.simulation_settings.equilibration_length)
print(f"New total simulation time: {new_total_sim_time} ns")

# Print the percentage increase in total simulation time
percentage_increase = ((new_total_sim_time - total_sim_time) / total_sim_time) * 100
print(f"Percentage increase in total simulation time: {percentage_increase.m:.2f}%")

Updated number of lambda windows: 22
New total simulation time: 396.0 nanosecond ns
Percentage increase in total simulation time: 100.00%
